<a href="https://colab.research.google.com/github/kevin305-dev/hello-world/blob/main/Retrieval_Retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrival

리트리버는 비정형 쿼리가 주어지면 **문서를 반환**하는 인터페이스다.
-  벡터 저장소 보다 더 일반적이다
-  문서를 저장할 필요 없이 단지 **반환**(또는 **검색**)만 할 수 있다 -- Langchain Document

>  쉽게 말해
Retriever는 **검색을 쉽게** 할 수 있도록 구성된 모듈이다.
-  이를 통해 손쉽게 **문서를 검색**할 수 있도록 하고
-  이를 기반으로 **LLM과 대화**할 수 있도록 한다.

In [1]:
!pip install chromadb tiktoken transformers sentence_transformers openai langchain pypdf

>  드라이브(drive)내 저장해둔 문서를 통해 RAG를 구성

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


>  openai chtgpt 모델을 통해 RAG 구성

In [3]:
import os
import openai
# os.environ["OPENAI_API_KEY"] = '-----'
openai.api_key = os.getenv("OPENAI_API_KEY")

## RAG 구성

>  token수 세기 함수
-  cl100k_base 이름을 가진 인코더를 기준으로 tokenizer를 선언
-  openai에서 임베딩 시킬때 쓰는 tokenizer 이름 중에 하나
-  이후, text를 chunking할때 **token 수를 세는 기준**을 **cl100k_base 모델**을 통해 세겠다는 의미

In [4]:
import tiktoken

tokenizer = tiktoken.get_encoding('cl100k_base')

def tiktoken_len(text):
  tokens = tokenizer.encode(text)
  return len(tokens)

>  사용하는 Library
-  RetrievalQA라는 chain 사용
-  ChatOpenAI를 통해 ChatGPT 3.5모델 사용
-  PyPDF Document Loader를 활용

In [8]:
!pip install -q langchain-text-splitters
!pip install -q langchain-community langchain-chroma pypdf langchain-huggingface

In [9]:
!pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 7.1 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.0
    Uninstalling langchain-1.4.0:
      Successfully uninstalled langchain-1.4.0


In [7]:
# RAG 체인 (구버전 스타일)
from langchain.chains import RetrievalQA
# LLM
from langchain_openai import ChatOpenAI
# 텍스트 분할
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 벡터스토어 (Chroma)
from langchain_community.vectorstores import Chroma
# 문서 로더
from langchain_community.document_loaders import PyPDFLoader
# -------------------------------------------------------------

ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# PyPDFLoader를 통해 문서를 로딩
loader = PyPDFLoader('/content/drive/MyDrive/IS-200_AI_Index_2025_시사점.pdf')
# PDF 파일을 page별로 자르고 list를 pages에 저장
pages = loader.load_and_split()

# 불러온 text를 RecursiveCharacterTextSplitter를 통해 chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = tiktoken_len
)
# document 객체의 형태(List)인 pages를 분할(splitting)
texts = text_splitter.split_documents(pages)

from langchain_huggingface import HuggingFaceEmbeddings

# 한글 Text Embedding에 유용한 모델
model_name = "jhgan/ko-sbert-nli"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {nomalize_embeddings: True}
hf = HuggingFaceEmbeddings(
    model_name = model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs
)

# 나뉜 것들에 대해 HuggingFace 모델을 적용해 Embedding 벡터로 변환하는 과정을 거치로
# Chroma Vectorstore에 저장
docsearch = Chroma.from_documents(text, hf)

>  RetrievalQA 선언

In [1]:
#from langchain.callbacks.streaming_stdout import StreamStdOutCallbackHandler
from langchain_core.callbacks import StreamingStdOutCallbackHandler

# streaming = True : 타자기처럼 결과가 출력
# temperature 0~2, 0일수록 답변이 일관적임
openai = ChatOpenAI(
    model_name = 'gpt-3.5-turbo',
    streaming = True,
    callbacks = [StreamingStdOutCallbackHandler()],
    temperature = 0
)

# gpt-3.5의 max token 4000
# chunk_size를 500으로 짧게 설정 및 gpt의 max token수에 제한을 두기위해 stuff 형태로 지정
# 참고) chunk_size가 크고, 질문도 길게 할수 있으면, Map reduce로 chain_type을 설정
#
# RAG를 구성할때 검색하는 건 뭘로 할래?=> docsearch를 as_retriever를 통해 검색기로 만듦
# Chroma 객체를 쓸건데, vector저장소에서 연관성 높은 벡터(text chunk)를 찾는 검색기로 쓸거야(as_retriever)
# search_type = 'mmr' : 연관성 높은 Pool 몇개 중에서 최대한 다양하게 답변을 context를 조합해서 LLM에게 context로 던져줘라
# fetch_k 여러개 연관문서 후보군의 총 갯수 10개를 뽑고 k(3)개 만큼만 LLM에 넘김
# return_source_documents : 질문에 대한 답변을 할때 LLM이 참고한 context가 어디에 있었던 문서인지 소스를 알수 있게 제공

qa = RetrievalQA.from_chain_type(
    llm = openai,
    chain_type = 'stuff',
    retriever = docsearch.as_retriever(
        search_type = 'mmr',
        search_kwargs = {'k':3, 'fetch_k': 10}
    ),
    return_source_documents = True
)

query = "혁신성장 정책금융에 대해서 설명해줘"
result = qa(query)

In [ ]:
result

결과)
혁신성장 정책금융은 기업의 성장과 혁신을 지원하기 위해 정부나 정부 기관이 제공하는 금융 지원 정책입니다. 이 정책은 .......

## Next :
1.   RAG 품질 향상 방법
2.   Streamlit/Chainlit으로 플랫폼 구축하여 ChatGPT와 대화하는 시스템 구축

##